# Train on Kaggle

Same training as `src/`, self-contained for Kaggle. Uploads `test-model.gguf` to the Cloud Storage bucket, then deploy from GitHub Actions.

**Before running (right panel):**
1. **Accelerator:** GPU T4 x2
2. **Internet:** On
3. **Input > Upload:** a dataset containing `datasets.jsonl` (from `data/raw/`)
4. **Add-ons > Secrets:** add `GCP_UPLOAD_KEY` (full JSON key of the `kaggle-upload` service account) and tick it
5. Set `BUCKET` in the next cell. Leave it empty to only save the model to `/kaggle/working`

**Run:** Save Version > Save & Run All runs it in the background (~1.5-2 h).
**Then:** GitHub > Actions > Deploy > Run workflow.

In [ ]:
BUCKET = ""

CONFIG = {
    "model_name": "test-model",
    "base_model": "unsloth/Qwen3-4B",
    "max_seq_length": 4096,
    "epochs": 2,
    "replay_samples": 150,
    "myanmar_samples": 300,
    "own_repeat": 3,
    "system_prompt": "You are a helpful AI assistant. Give clear, accurate answers. Be concise for simple questions and detailed when needed. Use markdown and code blocks for code.",
}

In [ ]:
%pip install -q unsloth

## 1. Load the base model

In [ ]:
import glob
import json
import os
import random
import shutil

from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

from datasets import Dataset, load_dataset
from trl import SFTConfig, SFTTrainer

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["base_model"],
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=True,
)


def generate(question, max_new_tokens=2048):
    messages = [{"role": "system", "content": CONFIG["system_prompt"]}, {"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.6, top_p=0.95, top_k=20, do_sample=True)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=False).split("<|im_end|>")[0].strip()
    return text, len(new_tokens) < max_new_tokens

## 2. Build the training set
Own examples with and without the system prompt, replay answers from the base model so it keeps its knowledge and reasoning, human-written Myanmar Q&A from `chuuhtetnaing/myanmar-aya-dataset` and Burmese instructions from `amkyawdev/myanmar-saillab-alpaca-myanmar-burmese-cleaned-instruction`.

In [ ]:
def with_system(messages):
    if messages[0]["role"] != "system":
        messages = [{"role": "system", "content": CONFIG["system_prompt"]}] + messages
    return {"messages": messages}


files = glob.glob("/kaggle/input/**/datasets.jsonl", recursive=True)
assert files, "datasets.jsonl not found - add it under Input"

own = []
with open(files[0], encoding="utf-8") as f:
    for line_number, line in enumerate(f, 1):
        if not line.strip():
            continue
        messages = json.loads(line)["messages"]
        assert any(m["role"] == "assistant" for m in messages), f"line {line_number}: no assistant message"
        own.append(with_system(messages))
assert own, "datasets.jsonl is empty"
own_without_system = [{"messages": example["messages"][1:]} for example in own]

FastLanguageModel.for_inference(model)
third = CONFIG["replay_samples"] // 3
general = load_dataset("databricks/databricks-dolly-15k", split="train").shuffle(seed=3407).select(range(third))
code = load_dataset("sahil2801/CodeAlpaca-20k", split="train").shuffle(seed=3407).select(range(third))
math_rows = load_dataset("openai/gsm8k", "main", split="train").shuffle(seed=3407).select(range(CONFIG["replay_samples"] - 2 * third))
questions = [f"{row['context']}\n\n{row['instruction']}" if row["context"] else row["instruction"] for row in general]
questions += [f"{row['instruction']}\n\n{row['input']}" if row["input"] else row["instruction"] for row in code]
questions += [row["question"] for row in math_rows]

replay = []
for i, question in enumerate(questions, 1):
    answer, finished = generate(question)
    if finished:
        replay.append(with_system([{"role": "user", "content": question}, {"role": "assistant", "content": answer}]))
    if i % 10 == 0:
        print(f"{i}/{len(questions)} answered, {len(replay)} kept")

pairs = [(row["inputs"], row["targets"]) for row in load_dataset("chuuhtetnaing/myanmar-aya-dataset", split="train")]
alpaca = load_dataset("amkyawdev/myanmar-saillab-alpaca-myanmar-burmese-cleaned-instruction", split="train")
for row in alpaca.shuffle(seed=3407).select(range(CONFIG["myanmar_samples"])):
    context = "" if row["input"].strip() == "nan" else row["input"].strip()
    pairs.append((f"{row['instruction'].strip()}\n\n{context}" if context else row["instruction"], row["output"]))

myanmar = [
    with_system([{"role": "user", "content": question.strip()}, {"role": "assistant", "content": answer.strip()}])
    for question, answer in pairs
    if question.strip() and answer.strip()
]

examples = (own + own_without_system) * CONFIG["own_repeat"] + replay + myanmar
random.Random(3407).shuffle(examples)
print(f"{len(own)} own (with and without system prompt) x {CONFIG['own_repeat']} + {len(replay)} replay + {len(myanmar)} Myanmar = {len(examples)} examples")

## 3. Train

In [ ]:
FastLanguageModel.for_training(model)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

dataset = Dataset.from_list(examples).map(
    lambda batch: {"text": [tokenizer.apply_chat_template(m, tokenize=False) for m in batch["messages"]]},
    batched=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=CONFIG["epochs"],
        learning_rate=2e-4,
        warmup_steps=5,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        output_dir="/tmp/checkpoints",
        save_strategy="no",
        report_to="none",
        seed=3407,
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

stats = trainer.train()
print(f"Took {stats.metrics['train_runtime'] / 60:.1f} min, final loss {stats.metrics['train_loss']:.3f}")

## 4. Test

In [ ]:
FastLanguageModel.for_inference(model)
for question in [
    "who created you?",
    "are you chatgpt?",
    "how do I reverse a string in python?",
    "A shop sells pens at 3 for $2. How much do 12 pens cost?",
    "မင်္ဂလာပါ၊ မင်းကို ဘယ်သူ ဖန်တီးခဲ့တာလဲ။",
]:
    print(f"Q: {question}\nA: {generate(question)[0]}\n")

## 5. Export GGUF
Exported in `/tmp` (more disk space), then copied to `/kaggle/working`.

In [ ]:
os.makedirs("/tmp/export", exist_ok=True)
os.chdir("/tmp/export")
model.save_pretrained_gguf(CONFIG["model_name"], tokenizer, quantization_method="q4_k_m")

matches = [p for p in glob.glob("/tmp/export/**/*.gguf", recursive=True) if "q4_k_m" in os.path.basename(p).lower()]
assert matches, "No Q4_K_M .gguf found - check the export log above"

GGUF = f"/kaggle/working/{CONFIG['model_name']}.gguf"
shutil.copy(matches[0], GGUF)
print(f"Saved {GGUF} ({os.path.getsize(GGUF) / 1e9:.2f} GB)")

## 6. Upload to Cloud Storage

In [ ]:
from google.cloud import storage
from kaggle_secrets import UserSecretsClient

if BUCKET:
    key = json.loads(UserSecretsClient().get_secret("GCP_UPLOAD_KEY"))
    blob = storage.Client.from_service_account_info(key).bucket(BUCKET).blob(os.path.basename(GGUF))
    blob.upload_from_filename(GGUF, timeout=3600)
    print(f"Uploaded to gs://{BUCKET}/{blob.name}")
    print("Now deploy it: https://github.com/NarihitoM/LLMTraining/actions/workflows/deploy.yml > Run workflow")
else:
    print("BUCKET not set, model is only in /kaggle/working")